# 01.분석

## 공통 확인
1. 행 수, 컬럼 수
2. 컬럼명
3. 데이터 타입
4. 샘플 데이터
5. 숫자형 컬럼의 최소값, 최대값, 평균값
6. 범주형 컬럼의 고유값 개수와 주요 값 분포
7. 컬럼별 결측치 개수
8. 컬럼별 결측률
9. 전체 중복 행 개수
10. 주요 식별자 기준 중복 개수
11. 문자열 앞뒤 공백
12. 한글 손상 문자 포함 여부
13. 숫자 컬럼에 문자열이 섞여 있는지
14. 날짜 형식이 일정한지
15. 위도/경도 형식과 범위가 정상인지
16. 시작일과 종료일
17. 연도별/월별 데이터 개수
18. 누락된 연도 또는 월
19. 동일 기간에 중복된 레코드
20. 최신 데이터 기준일
21. 지역 컬럼 존재 여부
22. 시도/시군구/주소/지사 중 어떤 단위인지
23. 지역별 데이터 개수
24. 지역명 표기 방식이 일관적인지
25. 주소에서 시도/시군구를 추출할 수 있는지
26. 지역 정보 결측률

## 01 가스 사고현황
1. 연도/월/가스구분 조합의 중복
2. 연도/월/가스구분 조합의 누락
3. 가스 유형별 사고 건수
4. 원인별 사고 건수
5. 사고 원인 합계가 0인 행
6. 지역 컬럼 부재 여부

In [3]:
import pandas as pd
import numpy as np

In [4]:
# Data/Raw/01_한국가스안전공사_가스사고현황(월별 원인별).csv
raw01 = pd.read_csv("../Data/Raw/01_한국가스안전공사_가스사고 현황(월별 원인별)_20251201.csv", encoding="cp949")

raw01.head(10)

,사고연도,사고월,가스구분,공급자취급부주의,기타(1-3급),사용자취급부주의,시설미비,제품노후(고장),타공사,교통사고
0,2021,1,LPG,0,1,3,0,2,0,0
1,2021,1,도시가스,0,1,0,0,1,1,0
2,2021,1,이동식부탄연소기(접합용기),0,0,5,0,0,0,0
3,2021,1,고압가스,0,0,0,0,1,0,0
4,2021,2,LPG,0,0,0,3,2,0,0
5,2021,2,도시가스,0,0,0,0,0,2,0
6,2021,2,이동식부탄연소기(접합용기),0,0,1,0,0,0,0
7,2021,3,LPG,0,0,0,0,1,1,0
8,2021,3,도시가스,0,0,0,2,0,0,0
9,2021,3,이동식부탄연소기(접합용기),0,0,1,0,0,0,0


In [14]:
# 1. 기본 구조 확인
df01 = raw01.copy()

print("행 수, 컬럼 수:", df01.shape)
print("컬럼명:", df01.columns.tolist())
print("데이터 타입:", df01.dtypes)

행 수, 컬럼 수: (158, 10)
컬럼명: ['사고연도', '사고월', '가스구분', '공급자취급부주의', '기타(1-3급)', '사용자취급부주의', '시설미비', '제품노후(고장)', '타공사', '교통사고']
데이터 타입: 사고연도        int64
사고월         int64
가스구분          str
공급자취급부주의    int64
기타(1-3급)    int64
사용자취급부주의    int64
시설미비        int64
제품노후(고장)    int64
타공사         int64
교통사고        int64
dtype: object


In [15]:
# 2. 결측치, 결측률 확인
missing_summary = pd.DataFrame({
    "missing_count": df01.isna().sum(),
    "missing_ratio": (df01.isna().mean() *100).round(2)
})

display(missing_summary)

,missing_count,missing_ratio
사고연도,0,0.0
사고월,0,0.0
가스구분,0,0.0
공급자취급부주의,0,0.0
기타(1-3급),0,0.0
사용자취급부주의,0,0.0
시설미비,0,0.0
제품노후(고장),0,0.0
타공사,0,0.0
교통사고,0,0.0


In [16]:
# 3. 전체 중복 행
print("전체 중복 행 수: ", df01.duplicated().sum())
display(df01[df01.duplicated(keep=False)])

전체 중복 행 수:  0


,사고연도,사고월,가스구분,공급자취급부주의,기타(1-3급),사용자취급부주의,시설미비,제품노후(고장),타공사,교통사고


In [20]:
# 4. 숫자 컬럼 품질 확인
numeric_cols = df01.select_dtypes(include="number").columns.tolist()

numeric_summary = pd.DataFrame({
    "mean": df01[numeric_cols].mean(),
    "std": df01[numeric_cols].std(),
    "min": df01[numeric_cols].min(),
    "25%": df01[numeric_cols].quantile(0.25),
    "50%": df01[numeric_cols].quantile(0.5),
    "75%": df01[numeric_cols].quantile(0.75),
    "max": df01[numeric_cols].max()
})
display(numeric_summary)

# 4. 음수값 확인
negative_summary = (df01[numeric_cols] < 0).sum().rename("negative_count")
print("음수값 개수:\n", negative_summary)

,mean,std,min,25%,50%,75%,max
사고연도,2022.879747,1.365441,2021,2022.0,2023.0,2024.0,2025
사고월,6.411392,3.414338,1,4.0,6.0,9.0,12
공급자취급부주의,0.234177,0.506867,0,0.0,0.0,0.0,3
기타(1-3급),0.208861,0.437913,0,0.0,0.0,0.0,2
사용자취급부주의,0.702532,0.848449,0,0.0,1.0,1.0,5
시설미비,0.487342,0.787945,0,0.0,0.0,1.0,3
제품노후(고장),0.398734,0.695205,0,0.0,0.0,1.0,3
타공사,0.227848,0.574387,0,0.0,0.0,0.0,4
교통사고,0.037975,0.191743,0,0.0,0.0,0.0,1


음수값 개수:
 사고연도        0
사고월         0
공급자취급부주의    0
기타(1-3급)    0
사용자취급부주의    0
시설미비        0
제품노후(고장)    0
타공사         0
교통사고        0
Name: negative_count, dtype: int64


In [21]:
# 5. 연도/월 범위와 분포
print("시작 연도:", df01["사고연도"].min())
print("종료 연도:", df01["사고연도"].max())
print("최소 월:", df01["사고월"].min())
print("최대 월:", df01["사고월"].max())

print("\n연도별 행 수:")
display(df01["사고연도"].value_counts().sort_index())

print("\n월별 행 수:")
display(df01["사고월"].value_counts().sort_index())

시작 연도: 2021
종료 연도: 2025
최소 월: 1
최대 월: 12

연도별 행 수:


사고연도
2021    33
2022    33
2023    37
2024    30
2025    25
Name: count, dtype: int64


월별 행 수:


사고월
1     16
2     11
3     10
4     14
5     14
6     16
7     15
8     13
9     13
10    13
11     9
12    14
Name: count, dtype: int64

In [26]:
# 6. 가스 유형 분포
gas_type_summary = (
    df01["가스구분"].value_counts(dropna=False).reset_index(name="count")
)
display(gas_type_summary)

,가스구분,count
0,LPG,53
1,도시가스,38
2,이동식부탄연소기(접합용기),35
3,고압가스,32


In [27]:
# 7. 연도/월/가스구분 조합 중복
key_cols = ["사고연도", "사고월", "가스구분"]
duplicate_keys = df01[df01.duplicated(key_cols, keep=False)]
print("연도/월/가스구분 조합 중복 행 수: ", len(duplicate_keys))
display(duplicate_keys.sort_values(key_cols))

연도/월/가스구분 조합 중복 행 수:  0


,사고연도,사고월,가스구분,공급자취급부주의,기타(1-3급),사용자취급부주의,시설미비,제품노후(고장),타공사,교통사고


In [29]:
# 8. 사고원인 합계
cause_cols = [
    "공급자취급부주의",
    "기타(1-3급)",
    "사용자취급부주의",
    "시설미비",
    "제품노후(고장)",
    "타공사",
    "교통사고",
]

df01["사고원인합계"] = df01[cause_cols].sum(axis=1)
display(df01.head(10))

,사고연도,사고월,가스구분,공급자취급부주의,기타(1-3급),사용자취급부주의,시설미비,제품노후(고장),타공사,교통사고,사고원인합계
0,2021,1,LPG,0,1,3,0,2,0,0,6
1,2021,1,도시가스,0,1,0,0,1,1,0,3
2,2021,1,이동식부탄연소기(접합용기),0,0,5,0,0,0,0,5
3,2021,1,고압가스,0,0,0,0,1,0,0,1
4,2021,2,LPG,0,0,0,3,2,0,0,5
5,2021,2,도시가스,0,0,0,0,0,2,0,2
6,2021,2,이동식부탄연소기(접합용기),0,0,1,0,0,0,0,1
7,2021,3,LPG,0,0,0,0,1,1,0,2
8,2021,3,도시가스,0,0,0,2,0,0,0,2
9,2021,3,이동식부탄연소기(접합용기),0,0,1,0,0,0,0,1


In [31]:
# 10. 원인별 전체 사고 건수
cause_summary = (
    df01[cause_cols].sum().sort_values(ascending=False).reset_index().rename(columns={"index" :"cause"})
)
display(cause_summary)

,cause,0
0,사용자취급부주의,111
1,시설미비,77
2,제품노후(고장),63
3,공급자취급부주의,37
4,타공사,36
5,기타(1-3급),33
6,교통사고,6


In [32]:
# 11. 가스구분별 전체 사고 건수
gas_accident_summary = (
    df01.groupby("가스구분", as_index=False)["사고원인합계"].sum().sort_values("사고원인합계", ascending=False)
)
display(gas_accident_summary)

,가스구분,사고원인합계
0,LPG,185
2,도시가스,70
3,이동식부탄연소기(접합용기),60
1,고압가스,48


In [33]:
# 12. 연도별 사고 건수 추이
monthly_summary = (
    df01.groupby(["사고연도", "사고월"], as_index=False)["사고원인합계"]
    .sum()
    .sort_values(["사고연도", "사고월"])
)

yearly_summary = (
    df01.groupby("사고연도", as_index=False)["사고원인합계"]
    .sum()
    .sort_values("사고연도")
)

display(monthly_summary.head(20))
display(yearly_summary)

,사고연도,사고월,사고원인합계
0,2021,1,15
1,2021,2,8
2,2021,3,5
3,2021,4,4
4,2021,5,6
5,2021,6,6
6,2021,7,6
7,2021,8,9
8,2021,9,1
9,2021,10,2


,사고연도,사고원인합계
0,2021,78
1,2022,73
2,2023,92
3,2024,70
4,2025,50


## 02. 가스사고 신고 접수 현황

In [45]:
raw02 = pd.read_csv("../Data/Raw/02_한국가스안전공사_ 가스사고_신고_접수_현황_20260323.csv", encoding="cp949")
df02 = raw02.copy()
print("행 수, 컬럼 수:", df02.shape)
print("컬럼명:", df02.columns.tolist())
print("데이터 타입:", df02.dtypes)

행 수, 컬럼 수: (16778, 10)
컬럼명: ['연번', '사고번호', '접수지사', '접수일자', '접수요일', '처리지사', '사고일자', '신고자 소속', '주소', '사고가스']
데이터 타입: 연번        int64
사고번호        str
접수지사        str
접수일자        str
접수요일        str
처리지사        str
사고일자        str
신고자 소속      str
주소          str
사고가스        str
dtype: object


In [ ]:
# 2. 결측치, 결측률 확인
missing_summary_02 = pd.DataFrame({
    "missing_count": df02.isna().sum(),
    "missing_ratio": (df02.isna().mean() *100).round(2)
})
display(missing_summary_02)

,missing_count,missing_ratio
연번,0,0.00
사고번호,0,0.00
접수지사,0,0.00
접수일자,0,0.00
접수요일,0,0.00
처리지사,0,0.00
사고일자,0,0.00
신고자 소속,0,0.00
주소,4,0.02
사고가스,10,0.06


In [47]:
# 3. 전체 중복 행, 사고번호 중복
print("전체 중복 행 수: ", df02.duplicated().sum())
print("사고번호 중복 수:", df02["사고번호"].duplicated().sum())

display(df02[df02["사고번호"].duplicated(keep=False)]
    )

전체 중복 행 수:  0
사고번호 중복 수: 0


,연번,사고번호,접수지사,접수일자,접수요일,처리지사,사고일자,신고자 소속,주소,사고가스


In [48]:
# 4. 날짜 형식과 기간
df02["접수일자"] = pd.to_datetime(df02["접수일자"], errors="coerce")
df02["사고일자"] = pd.to_datetime(df02["사고일자"], errors="coerce")

print("접수일자 범위:", df02["접수일자"].min(), "~", df02["접수일자"].max())
print("사고일자 범위:", df02["사고일자"].min(), "~", df02["사고일자"].max())

접수일자 범위: 2021-01-01 00:00:00 ~ 2026-02-28 00:00:00
사고일자 범위: 2020-09-05 00:00:00 ~ 2026-02-28 00:00:00


In [49]:
# 접수지사 - 처리지사 비교
different_branch = df02["접수지사"] != df02["처리지사"]

print("접수지사와 처리지사가 다른 행:", different_branch.sum())
print("비율(%):", round(different_branch.mean() * 100, 2))

접수지사와 처리지사가 다른 행: 16099
비율(%): 95.95


In [51]:
# 연도별 월별 접수 건수
df02["접수연도"] = df02["접수일자"].dt.year
df02["접수연월"] = df02["접수일자"].dt.to_period("M")

print("연도별 접수 건수:")
display(df02["접수연도"].value_counts().sort_index())

print("월별 접수 건수:")
display(
    df02["접수연월"]
    .value_counts()
    .sort_index()
    .rename_axis("접수연월")
    .reset_index(name="count")
)

연도별 접수 건수:


접수연도
2021    2643
2022    2137
2023    2573
2024    3942
2025    4604
2026     879
Name: count, dtype: int64

월별 접수 건수:


,접수연월,count
0,2021-01,189
1,2021-02,192
2,2021-03,227
3,2021-04,240
4,2021-05,223
...,...,...
57,2025-10,430
58,2025-11,459
59,2025-12,502
60,2026-01,492


In [ ]:
# 주소 결측과 지역 추출 가능 비율
print("주소 결측 수:", df02["주소"].isna().sum())
print("주소 결측률(%):", round(df02["주소"].isna().mean() * 100, 4))

address_parts = df02["주소"].fillna("").str.strip().str.split()

df02["추정_시도"] = address_parts.str[0]
df02["추정_시군구"] = address_parts.str[1]

extractable = (
    df02["추정_시도"].ne("")
    & df02["추정_시군구"].notna()
)

print("시도·시군구 추출 가능 행:", extractable.sum())
print("추출 가능 비율(%):", round(extractable.mean() * 100, 2))

주소 결측 수: 4
주소 결측률(%): 0.0238
시도·시군구 추출 가능 행: 16774
추출 가능 비율(%): 99.98


In [54]:
# 추정 지역 분포

print("시도별 신고 건수:")
display(
    df02["추정_시도"]
    .value_counts(dropna=False)
    .rename_axis("시도")
    .reset_index(name="count")
)

print("시군구별 신고 건수 TOP 30:")
display(
    df02.groupby(["추정_시도", "추정_시군구"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

시도별 신고 건수:


,시도,count
0,서울특별시,2012
1,경기도,1905
2,서울,1453
3,경기,1299
4,부산광역시,653
5,강원특별자치도,559
6,충청남도,518
7,부산,497
8,경상남도,483
9,인천광역시,473


시군구별 신고 건수 TOP 30:


,추정_시도,추정_시군구,count
532,충청북도,청주시,231
478,제주특별자치도,제주시,200
91,경기도,수원시,193
288,서울특별시,강남구,186
41,강원특별자치도,춘천시,158
519,충청남도,천안시,158
261,서울,강남구,157
505,충북,청주시,152
458,전북,전주시,143
476,제주,제주시,139


## 03 국내 가스시설 현황

In [56]:
raw03 = pd.read_csv("../Data/Raw/03_한국가스안전공사_국내 가스시설 현황_20250919.csv", encoding="cp949")
df03 = raw03.copy()
print("행 수, 컬럼 수:", df03.shape)
print("컬럼명:", df03.columns.tolist())
print("데이터 타입:", df03.dtypes)


행 수, 컬럼 수: (17, 25)
컬럼명: ['시도지역', '고압가스특정제조', '고압가스일반제조', '고압가스냉동제조', '고압가스충전', '고압가스저장', '고압가스특수독성제조', '고압가스특수독성저장', '고압가스일반판매', '고압가스일반LPG판매', '고압가스용기및용기부속품', '고압가스냉동기및특정설비제조', '고압가스사용시설', 'LPG충전', 'LPG저장', 'LPG집단공급사업자', 'LPG집단공급업소', 'LPG판매', 'LPG가스용품제조', 'LPG사용시설(정기검사)', 'LPG사용시설(면제대상)', '도시가스사업자', '도시가스대행', '도시가스사용시설', '가스시공업소']
데이터 타입: 시도지역                str
고압가스특정제조          int64
고압가스일반제조          int64
고압가스냉동제조          int64
고압가스충전            int64
고압가스저장            int64
고압가스특수독성제조        int64
고압가스특수독성저장        int64
고압가스일반판매          int64
고압가스일반LPG판매       int64
고압가스용기및용기부속품      int64
고압가스냉동기및특정설비제조    int64
고압가스사용시설          int64
LPG충전             int64
LPG저장             int64
LPG집단공급사업자        int64
LPG집단공급업소         int64
LPG판매             int64
LPG가스용품제조         int64
LPG사용시설(정기검사)     int64
LPG사용시설(면제대상)     int64
도시가스사업자           int64
도시가스대행            int64
도시가스사용시설          int64
가스시공업소            int64
dtype: object


In [57]:
# 결측치 와 결측률 
missing_summary_03 = pd.DataFrame({
    "missing_count": df03.isna().sum(),
    "missing_ratio": (df03.isna().mean() *100).round(2)
})
display(missing_summary_03)

,missing_count,missing_ratio
시도지역,0,0.0
고압가스특정제조,0,0.0
고압가스일반제조,0,0.0
고압가스냉동제조,0,0.0
고압가스충전,0,0.0
고압가스저장,0,0.0
고압가스특수독성제조,0,0.0
고압가스특수독성저장,0,0.0
고압가스일반판매,0,0.0
고압가스일반LPG판매,0,0.0


In [59]:
print("시도 개수:", df03["시도지역"].nunique())
print("시도 목록:")
print(sorted(df03["시도지역"].dropna().unique()))

시도 개수: 17
시도 목록:
['강원', '경기', '경남', '경북', '광주', '대구', '대전', '부산', '서울', '세종', '울산', '인천', '전남', '전북', '제주', '충남', '충북']


In [61]:
numeric_cols = df03.select_dtypes(include="number").columns.tolist()

numeric_summary = df03[numeric_cols].agg(["min", "max", "mean", "sum"]).T
display(numeric_summary)

negative_summary = (
    (df03[numeric_cols] < 0)
    .sum()
    .rename("negative_count")
    .to_frame()
)

display(negative_summary)

,min,max,mean,sum
고압가스특정제조,0.0,289.0,46.117647,784.0
고압가스일반제조,0.0,425.0,80.411765,1367.0
고압가스냉동제조,0.0,4674.0,946.529412,16091.0
고압가스충전,0.0,921.0,221.470588,3765.0
고압가스저장,0.0,1549.0,358.941176,6102.0
고압가스특수독성제조,0.0,10.0,0.764706,13.0
고압가스특수독성저장,0.0,102.0,23.176471,394.0
고압가스일반판매,0.0,236.0,40.529412,689.0
고압가스일반LPG판매,0.0,314.0,87.941176,1495.0
고압가스용기및용기부속품,0.0,22.0,5.176471,88.0


,negative_count
고압가스특정제조,0
고압가스일반제조,0
고압가스냉동제조,0
고압가스충전,0
고압가스저장,0
고압가스특수독성제조,0
고압가스특수독성저장,0
고압가스일반판매,0
고압가스일반LPG판매,0
고압가스용기및용기부속품,0


In [62]:
df03["전체시설수"] = df03[numeric_cols].sum(axis=1)

region_total = (
    df03[["시도지역", "전체시설수"]]
    .sort_values("전체시설수", ascending=False)
)

display(region_total)

,시도지역,전체시설수
7,경기,283053
14,경남,144532
2,대구,116908
8,강원,106069
9,충북,99154
13,경북,95641
12,전남,78552
10,충남,77085
11,전북,73382
5,대전,73008


In [63]:
facility_type_total = (
    df03[numeric_cols]
    .sum()
    .sort_values(ascending=False)
    .rename("시설수")
    .reset_index()
    .rename(columns={"index": "시설유형"})
)

display(facility_type_total)

,시설유형,시설수
0,LPG사용시설(면제대상),1110838
1,LPG사용시설(정기검사),173339
2,도시가스사용시설,104775
3,가스시공업소,19234
4,고압가스냉동제조,16091
5,고압가스저장,6102
6,고압가스사용시설,4755
7,고압가스충전,3765
8,LPG판매,2910
9,LPG집단공급업소,2091


In [64]:
high_pressure_cols = [
    col for col in numeric_cols
    if col.startswith("고압가스")
]

lpg_cols = [
    col for col in numeric_cols
    if col.startswith("LPG")
]

city_gas_cols = [
    col for col in numeric_cols
    if col.startswith("도시가스")
]

print("고압가스 컬럼:", high_pressure_cols)
print("LPG 컬럼:", lpg_cols)
print("도시가스 컬럼:", city_gas_cols)

고압가스 컬럼: ['고압가스특정제조', '고압가스일반제조', '고압가스냉동제조', '고압가스충전', '고압가스저장', '고압가스특수독성제조', '고압가스특수독성저장', '고압가스일반판매', '고압가스일반LPG판매', '고압가스용기및용기부속품', '고압가스냉동기및특정설비제조', '고압가스사용시설']
LPG 컬럼: ['LPG충전', 'LPG저장', 'LPG집단공급사업자', 'LPG집단공급업소', 'LPG판매', 'LPG가스용품제조', 'LPG사용시설(정기검사)', 'LPG사용시설(면제대상)']
도시가스 컬럼: ['도시가스사업자', '도시가스대행', '도시가스사용시설']


In [65]:
df03["고압가스시설수"] = df03[high_pressure_cols].sum(axis=1)
df03["LPG시설수"] = df03[lpg_cols].sum(axis=1)
df03["도시가스시설수"] = df03[city_gas_cols].sum(axis=1)

category_summary = df03[
    [
        "시도지역",
        "고압가스시설수",
        "LPG시설수",
        "도시가스시설수",
        "전체시설수",
    ]
].sort_values("전체시설수", ascending=False)

display(category_summary)

,시도지역,고압가스시설수,LPG시설수,도시가스시설수,전체시설수
7,경기,9643,239020,29783,283053
14,경남,3083,134628,5598,144532
2,대구,2320,107907,5588,116908
8,강원,924,102526,1781,106069
9,충북,2587,93158,2771,99154
13,경북,1247,91496,2086,95641
12,전남,2114,73522,2301,78552
10,충남,2549,70455,3430,77085
11,전북,1832,67492,3393,73382
5,대전,1581,66039,4589,73008


In [66]:
zero_facility_types = (
    df03[numeric_cols]
    .sum()
    .loc[lambda x: x == 0]
)

print("전국 합계가 0인 시설 유형:")
display(zero_facility_types)

전국 합계가 0인 시설 유형:


Series([], dtype: int64)

In [67]:
for _, row in df03.iterrows():
    top_types = (
        row[numeric_cols]
        .sort_values(ascending=False)
        .head(5)
    )

    print(f"\n[{row['시도지역']}]")
    display(top_types.rename("시설수").to_frame())


[서울]


,시설수
LPG사용시설(면제대상),30957
도시가스사용시설,23106
LPG사용시설(정기검사),5519
가스시공업소,4162
고압가스냉동제조,1577



[부산]


,시설수
LPG사용시설(면제대상),39726
도시가스사용시설,6662
LPG사용시설(정기검사),5118
가스시공업소,995
고압가스냉동제조,581



[대구]


...

,시설수
LPG사용시설(면제대상),117517
LPG사용시설(정기검사),16150
도시가스사용시설,5581
가스시공업소,1223
고압가스냉동제조,886



[제주]


,시설수
LPG사용시설(면제대상),51167
LPG사용시설(정기검사),7716
가스시공업소,522
LPG집단공급업소,227
고압가스냉동제조,152



[세종]


,시설수
고압가스특정제조,0
고압가스일반제조,0
도시가스사용시설,0
도시가스대행,0
도시가스사업자,0


In [68]:
concentration_summary = []

for col in numeric_cols:
    total = df03[col].sum()
    max_value = df03[col].max()
    max_region = df03.loc[df03[col].idxmax(), "시도지역"]

    concentration_summary.append({
        "시설유형": col,
        "전국합계": total,
        "최대지역": max_region,
        "최대지역시설수": max_value,
        "최대지역비율_percent": (
            round(max_value / total * 100, 2)
            if total > 0 else None
        ),
    })

display(
    pd.DataFrame(concentration_summary)
    .sort_values("최대지역비율_percent", ascending=False)
)

,시설유형,전국합계,최대지역,최대지역시설수,최대지역비율_percent
5,고압가스특수독성제조,13,울산,10,76.92
17,LPG가스용품제조,404,경기,202,50.00
0,고압가스특정제조,784,전남,289,36.86
7,고압가스일반판매,689,경기,236,34.25
10,고압가스냉동기및특정설비제조,1075,경기,366,34.05
1,고압가스일반제조,1367,경기,425,31.09
21,도시가스대행,272,서울,81,29.78
2,고압가스냉동제조,16091,경기,4674,29.05
22,도시가스사용시설,104775,경기,29717,28.36
6,고압가스특수독성저장,394,경기,102,25.89


## 04. 수소충전소 현황

In [1]:
import numpy as np
import pandas as pd